# NUTDTS 816 Time Series Analysis
## L09 Seasonal ARIMA

Lab notebook for Chapter 5 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 5.4 Worked example I: the airline model

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import tsdata
ap = tsdata.airpassengers(); y = np.log(ap)
train, test = y[:'1958-12'], y['1959-01':]     # hold out the last two years
d = train.diff().diff(12).dropna()
fig, axes = plt.subplots(1, 3, figsize=(11, 3), gridspec_kw={'width_ratios': [2, 1, 1]})
d.plot(ax=axes[0], lw=0.9, title='(1−B)(1−B¹²) log passengers, training set'); axes[0].set_xlabel('')
plot_acf(d, lags=36, ax=axes[1], title='ACF'); plot_pacf(d, lags=36, ax=axes[2], title='PACF')
for ax in axes[1:]: ax.set_ylim(-0.6, 1)
_caption = 'After one ordinary and one seasonal difference: a single negative spike at lag 1 and another at lag 12 in the ACF, with the PACF decaying at the seasonal lags. The signature of MA(1) × seasonal MA(1).'

In [ ]:
cands = [((0,1,1),(0,1,1,12)), ((1,1,0),(1,1,0,12)), ((0,1,1),(1,1,0,12)), ((1,1,0),(0,1,1,12)), ((0,1,2),(0,1,1,12)), ((1,1,1),(0,1,1,12))]
rows = []
for o, so in cands:
    f = ARIMA(train, order=o, seasonal_order=so).fit()
    lb = acorr_ljungbox(f.resid[13:], lags=[24], model_df=sum(o[::2]) + sum(so[:3:2]), return_df=True)
    rows.append({'order': f'{o}{so[:3]}12', 'AICc': round(f.aicc, 1), 'BIC': round(f.bic, 1), 'LB(24) p': round(lb.lb_pvalue.iloc[0], 3)})
print(pd.DataFrame(rows).sort_values('AICc').to_string(index=False))

In [ ]:
air = ARIMA(train, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit()
print(air.summary().tables[1])
fig = air.plot_diagnostics(figsize=(10, 5.5))
_caption = 'Airline model diagnostics on the training set: residuals are close to white noise and roughly normal.'

In [ ]:
h = len(test)
fc = air.get_forecast(h); ci = fc.conf_int(alpha=0.2)
snaive = pd.Series(train[-12:].values.tolist() * 2, index=test.index)     # seasonal naive in logs
ax = np.exp(y['1955':]).plot(figsize=(9, 3.4), lw=1, label='observed (incl. 1959-60 hold-out)')
np.exp(fc.predicted_mean).plot(ax=ax, color='#B8860B', lw=2, label='airline model')
ax.fill_between(ci.index, np.exp(ci.iloc[:, 0]), np.exp(ci.iloc[:, 1]), color='#B8860B', alpha=0.2, label='80% PI')
np.exp(snaive).plot(ax=ax, color='#555555', lw=1.2, ls='--', label='seasonal naive')
ax.axvline(test.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('Two-year forecast of airline passengers from December 1958')
act = np.exp(test)
for name, f_ in [('Airline model', np.exp(fc.predicted_mean)), ('Seasonal naive', np.exp(snaive))]:
    print(f'{name:16s} MAE = {(f_ - act).abs().mean():6.1f}   RMSE = {np.sqrt(((f_ - act)**2).mean()):6.1f}   (thousand passengers)')
_caption = 'The airline model tracks both the trend and the seasonal pattern over the two held-out years; the seasonal naive repeats 1958 and falls behind the growth.'

### 5.5 Worked example II: grid generation (simulated) and the STL-plus-ARIMA alternative

In [ ]:
from statsmodels.tsa.seasonal import STL
grid = tsdata.nigeria_grid(); gtr, gte = grid[:'2025-06'], grid['2025-07':]; h = len(gte)

# (a) SARIMA by automatic selection (pmdarima), then (b) STL + ARIMA, (c) seasonal naive
import pmdarima as pm
auto = pm.auto_arima(gtr, seasonal=True, m=12, stepwise=True, information_criterion='aicc', suppress_warnings=True, D=1)
print('auto_arima chose:', auto.order, auto.seasonal_order)
sar = ARIMA(gtr, order=auto.order, seasonal_order=auto.seasonal_order).fit()
f_sar = sar.get_forecast(h).predicted_mean

stl = STL(gtr, period=12, seasonal=13, robust=True).fit()
sa = gtr - stl.seasonal
auto_sa = pm.auto_arima(sa, seasonal=False, stepwise=True, information_criterion='aicc', suppress_warnings=True)
print('ARIMA on seasonally adjusted series:', auto_sa.order)
f_sa = ARIMA(sa, order=auto_sa.order, trend='t' if auto_sa.order[1] == 1 else 'c').fit().get_forecast(h).predicted_mean
seas_fwd = pd.Series(np.tile(stl.seasonal[-12:].values, 2)[:h], index=gte.index)    # last year's seasonal component carried forward
f_stl = f_sa + seas_fwd
f_sn = pd.Series(np.tile(gtr[-12:].values, 2)[:h], index=gte.index)

ax = grid['2022':].plot(figsize=(9, 3.4), lw=1, label='observed')
f_sar.plot(ax=ax, lw=2, color='#B8860B', label=f'SARIMA{auto.order}{auto.seasonal_order[:3]}12')
f_stl.plot(ax=ax, lw=1.5, color='#2F6DB5', ls='--', label='STL + ARIMA'); f_sn.plot(ax=ax, lw=1, color='#555555', ls=':', label='seasonal naive')
ax.axvline(gte.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('Grid generation (simulated): 12-month forecasts from June 2025')
for name, f_ in [('SARIMA', f_sar), ('STL + ARIMA', f_stl), ('Seasonal naive', f_sn)]:
    print(f'{name:16s} MAE = {(f_ - gte).abs().mean():6.0f} MW   RMSE = {np.sqrt(((f_ - gte)**2).mean()):6.0f} MW')
_caption = 'On a noisy series with a modest trend the ranking changes: STL + ARIMA is best, the seasonal naive is second, and the automatically selected SARIMA, whose seasonal AR(2) chases the last two years\' pattern, is worst. The collapses in the hold-out year are unforecastable by any of them.'

## Exercises

1. Expand SARIMA$(1,0,0)(1,0,0)_{12}$ into an explicit equation and identify the lag-13 term. Write the ACF at lags 1, 12 and 13 in terms of $\phi_1$ and $\Phi_1$.
2. Fit candidate SARIMA models to the log `a10` series (hold out the last 24 months). Shortlist from the ACF/PACF after $(1-B)(1-B^{12})$, compare by AICc, diagnose, forecast, and compare with seasonal naive and STL + ARIMA on the hold-out.
3. The `h02` series is monthly and has a strong seasonal pattern and a moderate trend. Compare $(d, D) = (0, 1)$ with $(1, 1)$ models: which has the better diagnostics, and what does the estimated seasonal MA coefficient tell you in each case?
4. Simulate two independent AR(1) series with $\phi = 0.95$ (stationary but persistent), regress one on the other, and report the $t$-statistic and Durbin-Watson. Is this spurious regression? What is different from the random-walk case, and what is the same?

In [ ]:
# Your work here
